In [1]:
import psycopg2
import pandas as pd
import networkx as nx
import base58
import matplotlib.pyplot as plt
import re
from math import comb
import matplotlib.dates as mdates
import networkx as nx
from collections import defaultdict
from networkx.algorithms.approximation.clique import large_clique_size
from itertools import combinations
import numpy as np
from networkx.algorithms.coloring import greedy_color
import random
from collections import Counter
import requests
import time
from datetime import datetime, timedelta, timezone

In [2]:
# Connect to PostgreSQL and fetch data
conn = psycopg2.connect(
    dbname="sui_indexer",
    user="postgres",
    password="56904628",
    host="172.26.112.1",
    port=5432
)
conn.autocommit = True
cur = conn.cursor()

In [3]:
cur.execute("""
SELECT
    t.checkpoint_sequence AS checkpoint,
    oc.transaction_digest AS tx,
    oc.address AS object_address,
    c.timestamp AS timestamp
FROM transactions t
JOIN object_changes oc
  ON oc.transaction_digest = t.transaction_digest
JOIN checkpoints c
  ON c.sequence_number = t.checkpoint_sequence
WHERE oc.change_type IS NOT NULL;
""")
write_rows = cur.fetchall()

In [4]:
# graphs[checkpoint] = nx.Graph()
graphs = defaultdict(nx.Graph)

# per checkpoint: object -> list of txs that wrote it
writers_by_cp_obj = defaultdict(lambda: defaultdict(list))

# all txs per checkpoint (so isolated nodes exist)
txs_by_cp = defaultdict(set)

# checkpoint -> timestamp
timestamps_by_cp = {}

for checkpoint, tx, object_address, timestamp in write_rows:
    tx_hex = tx.hex()
    obj_hex = object_address.hex()

    txs_by_cp[checkpoint].add(tx_hex)
    writers_by_cp_obj[checkpoint][obj_hex].append(tx_hex)

    # store timestamp
    if checkpoint not in timestamps_by_cp:
        timestamps_by_cp[checkpoint] = timestamp

# Build graphs
for cp, txs in txs_by_cp.items():
    G = graphs[cp]

    # 1) add all txs as nodes (isolated txs preserved)
    G.add_nodes_from(txs)

    # 2) add edges for shared-write objects
    for writers in writers_by_cp_obj[cp].values():
        writers = list(set(writers))  # de-dup
        if len(writers) < 2:
            continue

        for u, v in combinations(writers, 2):
            G.add_edge(u, v)

# graphs[checkpoint] = NetworkX conflict graph for that checkpoint
print(f"Built {len(graphs)} checkpoint graphs")

Built 73191 checkpoint graphs


In [5]:
df_8 = pd.read_csv('8M.csv')
df_33 = pd.read_csv('33M.csv')
df_66 = pd.read_csv('66M.csv')
df_150 = pd.read_csv('150M.csv')

In [6]:
df_all = pd.concat([df_8, df_33, df_66, df_150], ignore_index=True)

In [7]:
df_all["block_number"].min()

np.int64(8710000)

Density

In [8]:
rows = []

for cp, G in graphs.items():
    rows.append({
        "checkpoint": cp,
        "density": nx.density(G),
        "timestamp": timestamps_by_cp[cp]
    })

df_density = pd.DataFrame(rows)
df_density = df_density.set_index("checkpoint")

In [9]:
density = df_density.loc[138987740, "density"]
density

np.float64(0.16666666666666666)

In [10]:
median_density = df_density["density"].median()
print(f"The median density of our work is {median_density}")

The median density of our work is 0.09166666666666666


In [11]:
median_density_x = df_all["density"].median()
print(f"The median density of Biton's work is {median_density_x}")

The median density of Biton's work is 0.4705882352941176


Assortativity

In [12]:
assortativity = {}

for cp, G in graphs.items():
    # Skip empty or trivial graphs
    if G.number_of_nodes() < 2:
        assortativity[cp] = 0
        continue

    # DEGREE ASSORTATIVITY
    try:
        assortativity[cp] = nx.degree_assortativity_coefficient(G)
    except:
        assortativity[cp] = 0

c:\Users\Haygen Tsoi\.conda\envs\research_project\Lib\site-packages\networkx\algorithms\assortativity\correlation.py:302: RuntimeWarning: invalid value encountered in scalar divide
  return float((xy * (M - ab)).sum() / np.sqrt(vara * varb))
c:\Users\Haygen Tsoi\.conda\envs\research_project\Lib\site-packages\networkx\algorithms\assortativity\mixing.py:217: RuntimeWarning: invalid value encountered in divide
  a = a / a.sum()


In [13]:
df_assort = pd.DataFrame(
    [
        (cp, a, timestamps_by_cp[cp])
        for cp, a in assortativity.items()
    ],
    columns=["checkpoint", "assort", "timestamp"]
)

df_assort["timestamp"] = pd.to_datetime(df_assort["timestamp"], utc=True)


# Merge density + assortativity + block size
df_merge_ast = (
    df_density
    .merge(df_assort, on="checkpoint", how="inner")
)

In [14]:
df = df_merge_ast.dropna(subset=["assort", "density"])

# 1) Create density bins (e.g., 20 bins)
n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

# 2) Compute median assortativity per bin
per_bin_median = (
    df.groupby("density_bin")["assort"]
      .median()
      .dropna()
)

# 3) Final number: median of those medians
assort_median = per_bin_median.median()
print(f"The median assortativity at the same density of our work is {assort_median}")

The median assortativity at the same density of our work is 1.0


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\2916184376.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")
C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\2916184376.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["assort"]


In [15]:
df = df_all.copy()

df = df.dropna(subset=["assortativity", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["assortativity"]
      .median()
      .dropna()
)

assort_median = per_bin_median.median()
print(f"The median assortativity at the same density of Biton's work is {assort_median}")

The median assortativity at the same density of Biton's work is -0.499999999999998


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\1424468831.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["assortativity"]


Clique Number

In [37]:
clique_sizes = {}

for cp, G in graphs.items():
    if G.number_of_nodes() == 0:
        clique_sizes[cp] = 0
        continue

    # Fast greedy heuristic for max clique size
    clique_sizes[cp] = large_clique_size(G)


In [38]:
df_clique = pd.DataFrame(
    [
        (cp, c, timestamps_by_cp[cp])
        for cp, c in clique_sizes.items()
    ],
    columns=["checkpoint", "clique_size", "timestamp"]
)

df_clique["timestamp"] = pd.to_datetime(df_clique["timestamp"], utc=True)

df_merge_clique = (
    df_density
    .merge(df_clique, on="checkpoint", how="inner")
)

In [39]:
df = df_merge_clique.dropna(subset=["clique_size", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["clique_size"]
      .median()
      .dropna()
)

clique_median = per_bin_median.median()
print(f"The median clique number at the same density of our work is {clique_median}")

The median clique number at the same density of our work is 4.0


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\1595417956.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["clique_size"]


In [40]:
df = df_all.copy()

df = df.dropna(subset=["clique_number_approx", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["clique_number_approx"]
      .median()
      .dropna()
)

clique_median = per_bin_median.median()
print(f"The median clique number at the same density of Biton's work is {clique_median}")

The median clique number at the same density of Biton's work is 5.5


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\2912655575.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["clique_number_approx"]


Largest connected component

In [20]:
largest_cc = {}

for cp, G in graphs.items():
    if G.number_of_nodes() == 0:
        continue
    largest_cc[cp] = len(max(nx.connected_components(G), key=len))

In [21]:
df_lcc = pd.DataFrame(
    [
        (cp, lcc, timestamps_by_cp[cp])
        for cp, lcc in largest_cc.items()
    ],
    columns=["checkpoint", "largest_cc", "timestamp"]
)

df_lcc["timestamp"] = pd.to_datetime(df_lcc["timestamp"], utc=True)

# Merge density + largest_cc + block size
df_merge_lcc = (
    df_density
    .merge(df_lcc, on="checkpoint", how="inner")
)

In [22]:
df = df_merge_lcc.dropna(subset=["largest_cc", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["largest_cc"]
      .median()
      .dropna()
)

lcc_median = per_bin_median.median()
print(f"The median lcc at the same density of our work is {lcc_median}")

The median lcc at the same density of our work is 4.0


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\2682268570.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["largest_cc"]


In [25]:
df = df_all.copy()

df = df.dropna(subset=["largest_conn_comp", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["largest_conn_comp"]
      .median()
      .dropna()
)

lcc_median = per_bin_median.median()
print(f"The median lcc at the same density of Biton's work is {lcc_median}")

The median lcc at the same density of Biton's work is 8.5


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\757918718.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["largest_conn_comp"]


LSP/X lower bound

In [26]:
def greedy_long_path_len(C):
    start = max(C.degree, key=lambda x: x[1])[0]
    visited = {start}
    cur = start
    length = 0
    while True:
        nxts = [v for v in C.neighbors(cur) if v not in visited]
        if not nxts:
            break
        cur = min(nxts, key=lambda v: C.degree[v])
        visited.add(cur)
        length += 1
    return length

In [27]:
longest_simple_path = {}

for cp, G in graphs.items():
    length = greedy_long_path_len(G)

    longest_simple_path[cp] = length

df_path = pd.DataFrame(
    [
        (cp, lp, timestamps_by_cp[cp])
        for cp, lp in longest_simple_path.items()
    ],
    columns=["checkpoint", "longest_path", "timestamp"]
)

In [28]:
chromatic_number = {}

for cp, G in graphs.items():
    # Skip empty or trivial graphs
    if G.number_of_nodes() < 2:
        chromatic_number[cp] = 1 if G.number_of_nodes() == 1 else 0
        continue

    # CHROMATIC NUMBER (GREEDY APPROXIMATION)
    coloring = greedy_color(G, strategy="largest_first") 
    chromatic_number[cp] = len(set(coloring.values()))

df_chroma = pd.DataFrame(
    [
        (cp, c, timestamps_by_cp[cp])
        for cp, c in chromatic_number.items()
    ],
    columns=["checkpoint", "chromatic", "timestamp"]
)

In [29]:
# Merge: density + longest_path + chromatic + block size
df_ratio = (
    df_density
    .merge(df_path, on="checkpoint", how="inner")
    .merge(df_chroma, on="checkpoint", how="inner")
)

# Compute lower bound: longest_simple_path / greedy chromatic number
# Avoid division by zero; drop trivial graphs (<2 nodes) via NaN
df_ratio["ratio"] = np.where(
    df_ratio["chromatic"] > 0,
    df_ratio["longest_path"] / df_ratio["chromatic"],
    np.nan
)


In [30]:
df = df_ratio.dropna(subset=["ratio", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["ratio"]
      .median()
      .dropna()
)

lower_median = per_bin_median.median()
print(f"The median lsp/x lower bound at the same density of our work is {lower_median}")

The median lsp/x lower bound at the same density of our work is 0.6666666666666666


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\3829786180.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["ratio"]


In [31]:
df = df_all.copy()

df["ratio"] = np.where(
    df["greedy_color"] > 0,
    df["longest_path_length_monte_carlo"] / df["greedy_color"],
    np.nan
)

In [32]:
n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["ratio"]
      .median()
      .dropna()
)

lower_median = per_bin_median.median()
print(f"The median lsp/x lower bound at the same density of Biton's work is {lower_median}")

The median lsp/x lower bound at the same density of Biton's work is 1.2709131953318


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\3614435238.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["ratio"]


LSP/X upper bound

In [33]:
# Merge density + largest_cc + clique number + block size
df_upper = (
    df_density
    .merge(df_lcc, on="checkpoint", how="inner")
    .merge(df_clique, on="checkpoint", how="inner")
)

# Compute upper bound = largest CC / clique number
df_upper["upper_ratio"] = np.where(
    df_upper["clique_size"] > 0,
    df_upper["largest_cc"] / df_upper["clique_size"],
    np.nan
)

In [34]:
df = df_upper.dropna(subset=["upper_ratio", "density"])

n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["upper_ratio"]
      .median()
      .dropna()
)

upper_median = per_bin_median.median()
print(f"The median lsp/x upper bound at the same density of our work is {upper_median}")

The median lsp/x upper bound at the same density of our work is 1.0


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\2384232036.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["upper_ratio"]


In [35]:
df = df_all.copy()

df["upper_ratio"] = np.where(
    df["clique_number_approx"] > 0,
    df["largest_conn_comp"] / df["clique_number_approx"],
    np.nan
)

In [36]:
n_bins = 20
df["density_bin"] = pd.qcut(df["density"], q=n_bins, duplicates="drop")

per_bin_median = (
    df.groupby("density_bin")["upper_ratio"]
      .median()
      .dropna()
)

upper_median = per_bin_median.median()
print(f"The median lsp/x upper bound at the same density of Biton's work is {upper_median}")

The median lsp/x upper bound at the same density of Biton's work is 1.5


C:\Users\Haygen Tsoi\AppData\Local\Temp\ipykernel_11696\661539605.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("density_bin")["upper_ratio"]
